# Imports

In [ ]:
import io
import logging
import os
import re
import zipfile
from multiprocessing import Pool
from pathlib import Path
from pprint import pp, pprint
from typing import List, Tuple
import gc

import polars as pl
from tqdm import tqdm

# Global locks

In [ ]:
# Manifest mutex to prevent multiple processes from writing to the manifest at the same time
from filelock import FileLock

# Schemas

In [ ]:
OUTPUT_COLUMNS = [
    "source_id",
    "source_name",
    "mission",
    "dataset_family",
    "timestamp",
    "time_system",
    "density_kg_m3",
    "altitude_km",
    "latitude_deg",
    "longitude_deg",
    "local_solar_time_hours",
    "argument_of_latitude_deg",
    "quality_flag",
    "quality_detail",
    "native_file",
    "native_product",
    "native_record_index",
]
TUDELFT_SCHEMAS = {
    "CHAMP": {
        "Date yyyy-mm-dd": pl.Date,
        "Time hh:mm:ss.sss": pl.Time,
        "Time System": pl.Utf8,
        "Altitude (m)": pl.Float64,
        "Longitude (deg)": pl.Float64,
        "Latitude (deg)": pl.Float64,
        "Local Solar Time (hours)": pl.Float64,
        "Argument of Latitude (deg)": pl.Float64,
        "Density (kg/m^3)": pl.Float64,
        "Density Mean (kg/m^3)": pl.Float64,
        "Anomalus Density (kg/m^3)": pl.Float16,
        "Anomalus Density Mean (kg/m^3)": pl.Float16,
    },
    "GRACE": {
        "Date yyyy-mm-dd": pl.Date,
        "Time hh:mm:ss.sss": pl.Time,
        "Time System": pl.Utf8,
        "Altitude (m)": pl.Float64,
        "Longitude (deg)": pl.Float64,
        "Latitude (deg)": pl.Float64,
        "Local Solar Time (hours)": pl.Float64,
        "Argument of Latitude (deg)": pl.Float64,
        "Density (kg/m^3)": pl.Float64,
        "Density Mean (kg/m^3)": pl.Float64,
        "Anomalus Density (kg/m^3)": pl.Float16,
        "Anomalus Density Mean (kg/m^3)": pl.Float16,
    },
    "GRACE_FO": {
        "Date yyyy-mm-dd": pl.Date,
        "Time hh:mm:ss.sss": pl.Time,
        "Time System": pl.Utf8,
        "Altitude (m)": pl.Float64,
        "Longitude (deg)": pl.Float64,
        "Latitude (deg)": pl.Float64,
        "Local Solar Time (hours)": pl.Float64,
        "Argument of Latitude (deg)": pl.Float64,
        "Density (kg/m^3)": pl.Float64,
        "Density Mean (kg/m^3)": pl.Float64,
        "Anomalus Density (kg/m^3)": pl.Float16,
        "Anomalus Density Mean (kg/m^3)": pl.Float16,
    },
    "SWARM": {
        "Date yyyy-mm-dd": pl.Date,
        "Time hh:mm:ss.sss": pl.Time,
        "Time System": pl.Utf8,
        "Altitude (m)": pl.Float64,
        "Longitude (deg)": pl.Float64,
        "Latitude (deg)": pl.Float64,
        "Local Solar Time (hours)": pl.Float64,
        "Argument of Latitude (deg)": pl.Float64,
        "Density (kg/m^3)": pl.Float64,
        "Density Mean (kg/m^3)": pl.Float64,
        "Anomalus Density (kg/m^3)": pl.Float16,
        "Anomalus Density Mean (kg/m^3)": pl.Float16,
    },
    "GOCE": {
        "Date yyyy-mm-dd": pl.Date,
        "Time hh:mm:ss.sss": pl.Time,
        "Time System": pl.Utf8,
        "Altitude (m)": pl.Float64,
        "Longitude (deg)": pl.Float64,
        "Latitude (deg)": pl.Float64,
        "Local Solar Time (hours)": pl.Float64,
        "Argument of Latitude (deg)": pl.Float64,
        "Density (kg/m^3)": pl.Float64,
        "Density Mean (kg/m^3)": pl.Float64,
        "Anomalus Density (kg/m^3)": pl.Float16,
        "Anomalus Density Mean (kg/m^3)": pl.Float16,
        "Degraded Flag Thrusters": pl.Float16,
    },
}

# Decoder

## Decoder functions

In [ ]:
def normalize_whitespace(raw: bytes) -> bytes:
    return b"".join(
        b";".join(line.split()) + b"\n"
        for line in raw.splitlines()
        if line.strip() and not line.startswith(b"#")
    )


def decode_tudelft_single(
    mission: str,
    sourcepath: str,
    outfilepath: str,
    manifest_path: str,
    manifest_lock: FileLock,
) -> Tuple[str, str, str] | None:
    schema = TUDELFT_SCHEMAS[mission.upper()]
    parquet_path = str(Path(outfilepath).with_suffix(".parquet"))
    mission_code = str(Path(sourcepath).name).split("_")[0].upper()
    with zipfile.ZipFile(sourcepath, "r") as zip_ref:
        txt_name = next(
            (name for name in zip_ref.namelist() if name.lower().endswith(".txt")),
            None,
        )

        if txt_name is None:
            logging.warning("No .txt file found in %s", sourcepath)
            return None

        with zip_ref.open(txt_name, "r") as f:
            raw = f.read()

    fixed = normalize_whitespace(raw)
    try:
        df = pl.read_csv(
            io.BytesIO(fixed),
            separator=";",
            comment_prefix="#",
            has_header=False,
            schema=schema,
        )

        if df.is_empty() or df.width < 9:
            logging.warning(
                "Mission %s has an unexpected format and will be skipped.",
                sourcepath,
            )
            return None

        os.makedirs(os.path.dirname(parquet_path), exist_ok=True)
        df.write_parquet(parquet_path, compression="lz4")

        # Update manifest with locking
        with manifest_lock:
            if os.path.exists(manifest_path):
                manifest_df = pl.read_csv(manifest_path)
            else:
                manifest_df = pl.DataFrame(columns=["mission", "parquet_path"])

            new_entry = pl.DataFrame(
                {
                    "mission": [mission_code],
                    "parquet_path": [parquet_path],
                    "source_path": [sourcepath],
                }
            )
            updated_manifest = pl.concat([manifest_df, new_entry], how="vertical")
            updated_manifest.write_csv(manifest_path)

        return (mission, mission_code, parquet_path)
    except Exception as e:
        logging.error("Error processing %s: %s", sourcepath, e)
        print("\n".join(raw.decode("utf-8", errors="replace").splitlines()[:131]))
        print(
            "\n".join(
                [
                    # color double semicolons in red for better visibility
                    f"line: {i}, {len(line.split(';'))}, {line.replace(';;', '\033[91m;;\033[0m')}"
                    for i, line in enumerate(
                        fixed.decode("utf-8", errors="replace").splitlines()
                    )
                    if line[0] != "#" and len(line.split(";")) != len(schema)
                ]
            )
        )
        raise e
        exit(1)
        return None


def decode_tudelft_single_worker(args):
    return decode_tudelft_single(*args)


def merge_parquets(
    parquet_paths: List[str],
    output_path: str,
    manifest_path: str,
    manifest_lock: FileLock,
):
    dfs = []
    for path in tqdm(parquet_paths):
        try:
            df = pl.read_parquet(path)
            dfs.append(df)
        except Exception as e:
            logging.error("Error reading %s: %s", path, e)
    # create output directory if it doesn't exist
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    if dfs:
        merged_df = pl.concat(dfs, how="vertical")
        merged_df.write_parquet(output_path, compression="lz4")
        # Update manifest with locking
        with manifest_lock:
            if os.path.exists(manifest_path):
                manifest_df = pl.read_csv(manifest_path)
            else:
                manifest_df = pl.DataFrame(
                    columns=["mission_code", "merged_parquet_path"]
                )

            mission_code = Path(output_path).stem.split("_")[0]
            new_entry = pl.DataFrame(
                {
                    "mission_code": [mission_code],
                    "merged_parquet_path": [output_path],
                }
            )
            updated_manifest = pl.concat([manifest_df, new_entry], how="vertical")
            updated_manifest.write_csv(manifest_path)
    else:
        logging.warning("No valid parquet files to merge for %s", output_path)


def merge_parquets_single_worker(args):
    mission_code, parquet_paths, output_path, manifest_path, manifest_lock = args
    merge_parquets(parquet_paths, output_path, manifest_path, manifest_lock)

## Decoder runner

In [ ]:
inputpath: str = "data/original/tudelft"
outpath: str = "data/decoded/tudelft"
missions_list: List[str] = os.listdir(inputpath)

done_files = (
    pl.read_csv(outpath + "/tmp_decode_manifest.csv")["source_path"].to_list()
    if os.path.exists(outpath + "/tmp_decode_manifest.csv")
    else []
)

intermidiate_decode_manifest_lock = FileLock(outpath + "/tmp_decode_manifest.csv.lock")
arg_list = [
    (
        mission,
        inputpath + "/" + mission + "/" + filepath,
        outpath + "/" + mission + "/tmp/" + filepath.replace(".zip", ".parquet"),
        outpath + "/tmp_decode_manifest.csv",
        intermidiate_decode_manifest_lock,
    )
    for mission in missions_list
    for filepath in os.listdir(inputpath + "/" + mission)
    if filepath.lower().endswith(".zip")
    and inputpath + "/" + mission + "/" + filepath not in done_files
]
with Pool() as pool:
    results = [
        el
        for el in list(
            tqdm(
                pool.imap(
                    decode_tudelft_single_worker,
                    arg_list,
                ),
                total=len(arg_list),
            )
        )
        if el is not None
    ]
# one at a time for debugging
# results = [
#     decode_tudelft_single(*args) for args in tqdm(arg_list, total=len(arg_list))
# ]

if len(results) == 0:
    logging.warning("No valid files were decoded.")
if len(results) < len(arg_list):
    logging.warning(
        "%d out of %d files were decoded successfully.",
        len(results),
        len(arg_list),
    )

# group by mission code
mission_temp_paths = {}
for mission, mission_code, parquet_path in results:
    if mission_code not in mission_temp_paths:
        mission_temp_paths[mission_code] = []
    mission_temp_paths[mission_code].append(
        {"parquet_path": parquet_path, "mission": mission}
    )
pprint(mission_temp_paths)

decoded_manifest_path = outpath + "/decoded_manifest.csv"
if os.path.exists(decoded_manifest_path):
    decoded_manifest_sources = pl.read_csv(decoded_manifest_path)[
        "mission_code"
    ].to_list()
else:
    decoded_manifest_sources = []

decoded_manifest_lock = FileLock(outpath + "/decoded_manifest.csv.lock")
merge_args = [
    (
        mission_code,
        mission_temp_paths[mission_code],
        outpath
        + "/"
        + mission_temp_paths[mission_code][0]["mission"]
        + "/"
        + mission_code
        + "_merged.parquet",
        decoded_manifest_path,
        decoded_manifest_lock,
    )
    for mission_code in mission_temp_paths
    if mission_code not in decoded_manifest_sources
]
with Pool() as pool:
    pool.map(merge_parquets_single_worker, merge_args)
# one at a time for debugging
# for args in tqdm(merge_args, total=len(merge_args)):
#     merge_parquets_single_worker(args)

gc.collect()